# 🧠🤖 第二周-Day5：Transformer核心概念复习日

欢迎来到本周的复习日！🎉 今天我们要把四大核心概念串联起来，形成完整的知识脉络。就像拼图一样，把每个模块拼在一起，看看整幅Transformer图景有多美妙！

🔗 **本周知识脉络**：文字→词嵌入→注意力计算→网络层→整体架构

✨ **学习目标**：深入理解每个模块的作用和连接方式，为下周的大模型训练做好基础！

## 📚 Day1复习：FFN、LayerNorm与残差连接

这三个是Transformer的**训练稳定性保障**！没有它们，深度网络根本训练不起来。

🎯 **残差连接（Skip Connection）**：
• 像"抄近道"，让梯度直接流回🚀
• 解决深度网络梯度消失问题
• 公式：`y = x + F(x)` - 输入直接加到输出上

🎯 **LayerNorm（层归一化）**：
• 像"标准考试"，统一评分标准📏
• 归一化激活值，稳定训练
• 减少协变量偏移（Internal Covariate Shift）

🎯 **FFN（前馈网络）**：
• 像"思维加工厂"，处理信息🏭
• 公式：`FFN(x) = max(0, xW1 + b1)W2 + b2`
• 提供非线性变换能力

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 模拟残差连接的效果对比
def residual_connection(x, W1, b1, W2, b2):
    """带残差连接的FFN"""
    hidden = np.maximum(0, x @ W1 + b1)  # ReLU激活
    output = hidden @ W2 + b2           # 输出层
    return x + output                   # 残差连接

def normal_ffn(x, W1, b1, W2, b2):
    """普通FFN（无残差连接）"""
    hidden = np.maximum(0, x @ W1 + b1)
    return hidden @ W2 + b2

# 测试数据
np.random.seed(42)
x = np.random.randn(100, 512)
W1, b1 = np.random.randn(512, 2048), np.zeros(2048)
W2, b2 = np.random.randn(2048, 512), np.zeros(512)

# 测试不同深度的梯度保持
depths = range(1, 21)
residual_grads = []
normal_grads = []

for depth in depths:
    # 模拟多层网络
    input_x = x.copy()
    
    # 残差连接版本
    for _ in range(depth):
        input_x = residual_connection(input_x, W1, b1, W2, b2)
    residual_grads.append(np.linalg.norm(input_x))
    
    # 普通版本
    input_x = x.copy()
    for _ in range(depth):
        input_x = normal_ffn(input_x, W1, b1, W2, b2)
    normal_grads.append(np.linalg.norm(input_x))

# 可视化结果
plt.figure(figsize=(12, 6))
plt.plot(depths, residual_grads, 'b-', label='残差连接', linewidth=2)
plt.plot(depths, normal_grads, 'r--', label='普通FFN', linewidth=2)
plt.xlabel('网络层数')
plt.ylabel('输出范数')
plt.title('残差连接对深度网络梯度的影响')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 📚 Day2复习：Tokenizer与词嵌入

这是文字变成数学语言的**桥梁工程**！没有它们，计算机无法理解人类语言。

🎯 **Tokenizer分词器**：
• 像"切菜刀"，把文字切成小块🔪
• BPE算法动态合并频繁字符对
• 平衡词表大小和表达能力

🎯 **词嵌入（Embedding）**：
• 像"翻译官"，文字→向量🌐
• 300-768维稠密向量
• 捕捉语义关系（king - man + woman ≈ queen）

🎯 **位置编码（Positional Encoding）**：
• 像"座位表"，记住token位置🎫
```
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 简化的BPE分词演示
class SimpleBPE:
    def __init__(self):
        self.vocab = {'<|start|>': 0, '<|end|>': 1}
        self.merges = {}
        self.word_freqs = {}
        
    def get_tokens(self, word):
        """将单词拆分为字符"""
        return list(word)
    
    def train_bpe(self, corpus, num_merges=100):
        """简化版BPE训练"""
        # 初始化：将所有单词拆分为字符
        for word in corpus:
            chars = self.get_tokens(word)
            for i in range(len(chars) - 1):
                pair = (chars[i], chars[i+1])
                self.word_freqs[pair] = self.word_freqs.get(pair, 0) + 1
        
        print(f"初始词表大小: {len(self.vocab)}")
        print(f"初始合并候选: {list(self.word_freqs.keys())[:10]}")

# 示例训练
corpus = ["hello", "world", "hello", "transformer", "attention"]
bpe = SimpleBPE()
bpe.train_bpe(corpus)

# 词嵌入可视化
def visualize_embeddings():
    # 创建一些简单的词向量
    words = ['king', 'queen', 'man', 'woman', 'apple', 'orange']
    embeddings = np.random.randn(len(words), 50) * 0.1
    
    # 设置一些语义关系
    embeddings[0] += 0.5  # king
    embeddings[1] = embeddings[0] + 0.2  # queen ≈ king + something
    embeddings[2] = embeddings[0] - 0.3  # man ≈ king - something  
    embeddings[3] = embeddings[1] - 0.3  # woman ≈ queen - something
    
    # 计算相似度
    similarities = []
    for i in range(len(words)):
        for j in range(i+1, len(words)):
            sim = np.dot(embeddings[i], embeddings[j]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j]))
            similarities.append((words[i], words[j], sim))
    
    # 显示相似度
    print("词向量相似度：")
    for word1, word2, sim in sorted(similarities, key=lambda x: -x[2]):
        print(f"{word1} vs {word2}: {sim:.3f}")
    
    # 词向量散点图
    plt.figure(figsize=(10, 8))
    colors = ['red', 'red', 'blue', 'blue', 'green', 'green']
    for i, (word, color) in enumerate(zip(words, colors)):
        plt.scatter(embeddings[i, 0], embeddings[i, 1], 
                   color=color, s=100, label=word)
        plt.annotate(word, (embeddings[i, 0], embeddings[i, 1]), 
                    xytext=(5, 5), textcoords='offset points')
    
    plt.xlabel('维度 0')
    plt.ylabel('维度 1')
    plt.title('词向量2D投影（颜色表示语义类别）')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

visualize_embeddings()

## 📚 Day3复习：GPT vs BERT架构对比

这是Transformer的**两大应用方向**！一个像"说书人"，一个像"理解者"。

🎯 **GPT（生成式预训练）**：
• 像"说书人"，擅长生成内容🎭
• **单向注意力**：只能看到左边的token
• 自回归：`P(w_i | w_1, w_2, ..., w_{i-1})`
• 应用：写作、对话、代码生成

🎯 **BERT（双向编码器）**：
• 像"理解者"，擅长理解语义📖
• **双向注意力**：能看到上下文所有token
• 掩码语言模型：`P(w_i | context)`
• 应用：分类、问答、情感分析

🎯 **核心差异**：
• **注意力机制**：单向 vs 双向
• **训练目标**：生成 vs 理解
• **应用场景**：创作 vs 分析

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 可视化GPT和BERT的注意力差异
def visualize_attention_patterns():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # 创建示例句子
    sentence = ["我", "喜欢", "吃", "苹果", "和", "香蕉"]
    
    # GPT单向注意力（只能看左边）
    gpt_attention = np.zeros((len(sentence), len(sentence)))
    for i in range(len(sentence)):
        for j in range(i + 1):  # 只能看j <= i的位置
            gpt_attention[i, j] = 1.0 / (i - j + 1)  # 距离越近权重越大
    
    # BERT双向注意力（看上下文）
    bert_attention = np.zeros((len(sentence), len(sentence)))
    for i in range(len(sentence)):
        for j in range(len(sentence)):
            if i != j:
                distance = abs(i - j)
                bert_attention[i, j] = 1.0 / (distance + 1)
        
    # 绘制热力图
    im1 = ax1.imshow(gpt_attention, cmap='Blues', aspect='auto')
    ax1.set_title('GPT 单向注意力', fontsize=14, fontweight='bold')
    ax1.set_xlabel('当前位置')
    ax1.set_ylabel('注意力位置')
    ax1.set_xticks(range(len(sentence)))
    ax1.set_yticks(range(len(sentence)))
    ax1.set_xticklabels(sentence, rotation=45)
    ax1.set_yticklabels(sentence)
    ax1.grid(True, alpha=0.3)
    
    im2 = ax2.imshow(bert_attention, cmap='Reds', aspect='auto')
    ax2.set_title('BERT 双向注意力', fontsize=14, fontweight='bold')
    ax2.set_xlabel('当前位置')
    ax2.set_ylabel('注意力位置')
    ax2.set_xticks(range(len(sentence)))
    ax2.set_yticks(range(len(sentence)))
    ax2.set_xticklabels(sentence, rotation=45)
    ax2.set_yticklabels(sentence)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 打印一些对比信息
    print("🎯 GPT vs BERT 对比分析：")
    print("\n📊 GPT 单向注意力特点：")
    print(f"• 句子'苹果'只能关注: {[sentence[i] for i in range(3, 4)]}")
    print(f"• 句子'香蕉'只能关注: {[sentence[i] for i in range(5)]}")
    print("\n📊 BERT 双向注意力特点：")
    print(f"• 句子'苹果'可以关注: {sentence}")
    print(f"• 句子'香蕉'可以关注: {sentence}")
    
visualize_attention_patterns()

## 📚 Day4复习：NumPy实现Self-Attention

这是Transformer的**核心灵魂**！没有注意力机制，就没有Transformer的革命。

🎯 **Self-Attention计算步骤**：
1. **生成Query、Key、Value**：
   ```
   Q = X @ Wq, K = X @ Wk, V = X @ Wv
   ```
2. **计算注意力分数**：
   ```
   Attention(Q,K,V) = softmax(QK^T/√d_k)V
   ```
3. **加权求和**：
   ```
   Output = ∑(attention_score_i × Value_i)
   ```

🎯 **关键特点**：
• **并行计算**：所有token同时计算注意力
• **动态权重**：每个token关注其他token的权重不同
• **可学习参数**：Wq, Wk, Wv矩阵通过训练学习

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class SimpleSelfAttention:
    def __init__(self, d_model=64, d_k=64):
        self.d_model = d_model
        self.d_k = d_k
        # 初始化权重矩阵
        np.random.seed(42)
        self.Wq = np.random.randn(d_model, d_k) * 0.01
        self.Wk = np.random.randn(d_model, d_k) * 0.01
        self.Wv = np.random.randn(d_model, d_k) * 0.01
    
    def softmax(self, x):
        """稳定版本的softmax"""
        x_max = np.max(x, axis=-1, keepdims=True)
        exp_x = np.exp(x - x_max)
        sum_exp = np.sum(exp_x, axis=-1, keepdims=True)
        return exp_x / sum_exp
    
    def self_attention(self, X):
        """完整的Self-Attention计算"""
        batch_size, seq_len, d_model = X.shape
        
        # Step 1: 生成Q, K, V
        Q = np.dot(X, self.Wq)  # (batch, seq_len, d_k)
        K = np.dot(X, self.Wk)  # (batch, seq_len, d_k)
        V = np.dot(X, self.Wv)  # (batch, seq_len, d_k)
        
        # Step 2: 计算注意力分数
        scores = np.dot(Q, K.transpose(0, 2, 1))  # (batch, seq_len, seq_len)
        scores = scores / np.sqrt(self.d_k)      # 缩放防止梯度爆炸
        
        # Step 3: Softmax
        attention_weights = self.softmax(scores)
        
        # Step 4: 加权求和
        output = np.dot(attention_weights, V)  # (batch, seq_len, d_k)
        
        return output, attention_weights

# 测试Self-Attention
def test_self_attention():
    # 创建测试数据：一个简单的句子
    # tokens: 我 爱 你 机器 学习
    np.random.seed(42)
    X = np.random.randn(1, 5, 64)  # batch=1, seq_len=5, d_model=64
    
    attention = SimpleSelfAttention()
    output, attention_weights = attention.self_attention(X)
    
    print(f"输入shape: {X.shape}")
    print(f"输出shape: {output.shape}")
    print(f"注意力权重shape: {attention_weights.shape}")
    
    # 可视化注意力权重
    plt.figure(figsize=(10, 8))
    
    # 热力图
    plt.subplot(2, 2, 1)
    plt.imshow(attention_weights[0], cmap='Blues', aspect='auto')
    plt.colorbar()
    plt.title('注意力权重热力图')
    plt.xlabel('Key位置')
    plt.ylabel('Query位置')
    
    # 条形图显示每个token的注意力分布
    plt.subplot(2, 2, 2)
    tokens = ['我', '爱', '你', '机器', '学习']
    for i, token in enumerate(tokens):
        plt.bar(token, attention_weights[0, i, :].sum(), alpha=0.7)
    plt.title('各token总关注度')
    plt.ylabel('总注意力权重')
    
    # 显示具体的注意力矩阵
    plt.subplot(2, 1, 2)
    im = plt.imshow(attention_weights[0], cmap='viridis', aspect='auto')
    plt.colorbar()
    plt.title('详细的注意力矩阵')
    plt.xlabel('Key位置')
    plt.ylabel('Query位置')
    
    # 添加标签
    plt.xticks(range(len(tokens)), tokens, rotation=45)
    plt.yticks(range(len(tokens)), tokens)
    
    plt.tight_layout()
    plt.show()
    
    print("\n🔍 注意力分析：")
    for i, query_token in enumerate(tokens):
        print(f"\n'{query_token}' 的注意力分布：")
        attention_dist = attention_weights[0, i, :]
        for j, key_token in enumerate(tokens):
            print(f"  → 关注'{key_token}': {attention_dist[j]:.3f}")

test_self_attention()

## 🧩 综合练习与测试

### 📝 课堂练习（5分钟）

❶ **残差连接的作用是什么？**
   A. 增加模型参数量
   B. ✅ 解决深度网络梯度消失问题
   C. 提供非线性激活
   D. 减少计算量

❷ **BERT相比GPT的最大优势是什么？**
   A. 生成长文本能力强
   B. ✅ 能够同时看到上下文信息
   C. 参数量更小
   D. 训练速度更快

❸ **在Self-Attention中，为什么要除以√d_k？**
   A. 防止过拟合
   B. ✅ 防止注意力分数过大导致softmax梯度消失
   C. 加快收敛速度
   D. 减少内存使用

❹ **BPE分词相比传统分词的优势是什么？**
   A. 速度更快
   B. 处理未知词更好
   C. ✅ 平衡词表大小和表达能力
   D. 实现更简单

❺ **LayerNorm和BatchNorm的主要区别是什么？**
   A. LayerNorm对batch归一化，BatchNorm对特征归一化
   B. ✅ LayerNorm对单个样本的各特征归一化，BatchNorm对batch的各样本归一化
   C. LayerNorm更适合图像，BatchNorm更适合文本
   D. 没有区别，只是名称不同

**💡 提示**：重点关注每个模块的**设计目的**和**解决的问题**！

In [ ]:
# 自动批改练习
def check_answers():
    """检查答案并给出解析"""
    correct_answers = ['B', 'B', 'B', 'C', 'B']
    questions = [
        "残差连接的作用是什么？",
        "BERT相比GPT的最大优势是什么？",
        "在Self-Attention中，为什么要除以√d_k？",
        "BPE分词相比传统分词的优势是什么？",
        "LayerNorm和BatchNorm的主要区别是什么？"
    ]
    explanations = [
        "残差连接让梯度可以直接流回，避免了深度网络中梯度逐层衰减的问题。",
        "BERT使用双向注意力，可以同时看到左右两边的上下文信息，而GPT只能看到左边。",
        "除以√d_k可以防止注意力分数过大，避免softmax梯度消失。",
        "BPE通过合并频繁字符对，可以在有限词表大小下保持良好的表达能力。",
        "LayerNorm对单个样本的特征进行归一化，BatchNorm对batch中样本的特征进行归一化。"
    ]
    
    print("🎯 练习答案解析：")
    print("=" * 50)
    
    for i, (question, correct, explanation) in enumerate(zip(questions, correct_answers, explanations)):
        print(f"❷ {question}")
        print(f"   正确答案：{correct}")
        print(f"   解析：{explanation}")
        print()
    
    print("💡 复习建议：")
    print("• 重点理解每个模块的**设计目的**，而不仅仅是公式")
    print("• 画图帮助理解注意力机制的工作原理")
    print("• 自己动手实现一个小型的Self-Attention")

check_answers()

## 🔗 推荐资源

### 🎬 推荐视频：
1. **【2025新】这应该是B站最全面详细的Transformer教程!**
   - 链接：https://www.bilibili.com/video/BV1zaa2zfEX9/
   - 时长：12集完整教程，包含从理论到实战
   - 重点：第2集讲Tokenization，第3集讲Embedding

2. **还不会 BPE 分词？这个视频让你彻底搞懂!**
   - 链接：https://www.bilibili.com/video/BV1zBRzYQEzt/
   - 时长：约20分钟
   - 重点：BPE分词算法原理解析

### 📖 延伸阅读：
1. **LayerNorm与残差连接深入理解**
   - 链接：https://zhuanlan.zhihu.com/p/2030038660100383584
   - 重点：从深度网络训练困难角度理解LayerNorm和残差连接

2. **一文彻底搞懂Transformer - Add & Norm**
   - 链接：https://blog.csdn.net/2401_85377976/article/details/141423008
   - 重点：残差连接和层归一化的实现细节

3. **transformer结构-输入编码(BPE,PE)剖析**
   - 链接：https://zhuanlan.zhihu.com/p/667635031
   - 重点：Tokenization的深度解析

## 🎯 总结与思考

### 💡 知识卡片（本周精华）：

🎯 **残差连接**：
• 作用：解决梯度消失问题
• 公式：`y = x + F(x)`
• 意义：让信息能"跳过"多层直接传递

🎯 **LayerNorm**：
• 作用：稳定训练，减少协变量偏移
• 位置：每个子层（Self-Attention + FFN）之后
• 特点：对单个样本的特征进行归一化

🎯 **词嵌入**：
• 作用：文字→向量
• 特点：捕捉语义关系，300-768维
• 优势：表示能力强，适合深度学习

🎯 **注意力机制**：
• 作用：计算token之间的关联权重
• 类型：单向（GPT）vs 双向（BERT）
• 核心：`softmax(QK^T/√d_k)V`

### 🔄 下周预告：

🚀 **第三周：大模型训练全景**

• Day 1：预训练 - 数据、规模与涌现能力
• Day 2：SFT - 监督微调的关键细节
• Day 3：RLHF - 人类反馈强化学习
• Day 4：DPO与其他对齐方法
• Day 5：训练全流程串联
• Day 6：⚡ unsloth做QLoRA微调demo
• Day 7：🔄复习巩固

🎯 **准备建议**：
• 熟悉本周的Transformer基础
• 准备一些自己的技术文档用于下周RAG实践
• 思考业务场景中可能用到大模型的地方

In [ ]:
# 学习进度追踪
import matplotlib.pyplot as plt
import numpy as np

def show_progress():
    """显示学习进度"""
    weeks = ['W1', 'W2', 'W3', 'W4', 'W5', 'W6', 'W7', 'W8', 'W9', 'W10', 'W11', 'W12']
    progress = [0.33, 0.67, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  # W1=33%, W2=67%完成
    
    plt.figure(figsize=(12, 6))
    
    # 进度条
    bars = plt.bar(weeks, progress, color='skyblue', alpha=0.7, edgecolor='navy', linewidth=2)
    
    # 添加进度标签
    for bar, prog in zip(bars, progress):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{prog*100:.0f}%', ha='center', va='bottom', fontweight='bold')
    
    # 当前位置标记
    current_week = 1  # 当前处于第2周
    plt.axvline(x=current_week-0.5, color='red', linestyle='--', linewidth=2, alpha=0.7)
    plt.text(current_week-0.5, 0.8, '当前位置', rotation=90, va='top', color='red', fontweight='bold')
    
    plt.xlabel('学习周')
    plt.ylabel('完成进度')
    plt.title('12周大模型+脑科学学习进度')
    plt.ylim(0, 1.2)
    plt.grid(True, alpha=0.3)
    
    # 添加阶段说明
    plt.text(2.5, 1.1, '🤖 大模型基础', ha='center', fontsize=12, fontweight='bold')
    plt.text(8.5, 1.1, '🧠 脑科学基础', ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 当前进度：第{current_week}周/12周")
    print(f"🎯 大模型部分：W1-W6（已完成前2周的基础架构学习）")
    print(f"🎯 脑科学部分：W7-W12（待开始）")
    print(f"⏰ 下周预告：W3 大模型训练全景")

show_progress()